# EnergyCast V1 — Step 11: Non-Linear Gradient Boosted Trees (LightGBM & XGBoost)
## Electricity Price Forecasting System (Indian Electricity Market)

**Objective:** Train and rigorously benchmark state-of-the-art Gradient Boosted Decision Tree (GBDT) architectures—LightGBM (histogram-based leaf-wise) and XGBoost (exact second-order Newton)—against the Seasonal Naive and Ridge Regression baselines.  

### Key Highlights:
1. **Tree Ensembles**: Captures complex non-linearities, regime shifts, and multi-feature interactions (merit-order curve inflection, ramp stress, bidding deficit).
2. **Strict Leakage Prevention**: Features trained on historical lags ($t-24$) and physical exogenous grid/weather variables. Zero contemporaneous price or clearing volumes.
3. **Early Stopping Regularization**: Monitored on the 2024 Validation set with 50-round patience to completely avert overfitting.
4. **Out-of-Sample Breakthrough**: Evaluated on the unseen 2025 Test partition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
TREE_METRICS_PATH = Path('../outputs/tables/tree_models_metrics.csv')
FEAT_IMP_PATH = Path('../outputs/tables/tree_feature_importances.csv')
RIDGE_METRICS_PATH = Path('../outputs/tables/linear_ridge_metrics.csv')
SNAIVE_METRICS_PATH = Path('../outputs/tables/seasonal_naive_metrics.csv')
TEST_PREDS_PATH = Path('../outputs/predictions/trees_test_predictions.csv')

print('Tree Metrics Exists:', TREE_METRICS_PATH.exists())
print('Feature Importances Exists:', FEAT_IMP_PATH.exists())
print('Predictions Exists:', TEST_PREDS_PATH.exists())

### 1. Three-Tier Model Hierarchy Benchmark
Comparing Non-Parametric (Seasonal Naive), Linear Parametric (Ridge), and Non-Linear Trees (LightGBM & XGBoost) on the 2025 Test Set.

In [ ]:
tree_df = pd.read_csv(TREE_METRICS_PATH)
print("--- GBDT Validation and Test Performance ---")
display(tree_df)

# Cross-model comparison on Test Set
test_tree = tree_df[tree_df['Split'].str.contains('Test')].copy()
print("\nTest Set Performance Summary:")
for _, row in test_tree.iterrows():
    print(f"{row['Model']}: MAE = Rs {row['MAE']:.2f}, RMSE = Rs {row['RMSE']:.2f}, R2 = {row['R2']:.4f}")

### 2. Feature Importance Rankings (Split Gain vs. Frequency)
Examining the dominant price drivers identified by LightGBM and XGBoost.

In [ ]:
feat_imp = pd.read_csv(FEAT_IMP_PATH)
top_15 = feat_imp.head(15)
display(top_15)

# Bar plot of top 15 features by LightGBM Gain
fig, ax = plt.subplots(figsize=(10, 6))
norm_gain = top_15['LGBM_Gain'] / top_15['LGBM_Gain'].max() * 100.0
y_pos = np.arange(len(top_15))
ax.barh(y_pos, norm_gain.iloc[::-1], color='#2E75B6', alpha=0.85, height=0.65)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_15['Feature'].iloc[::-1])
ax.set_xlabel('Relative Gain (% of Top Feature: mcp_lag_24)')
ax.set_title('Top 15 Most Influential Features (LightGBM Total Gain)', fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Out-of-Sample Actual vs. Predicted Parity
Assessing forecast adherence across the entire price domain [Rs 0 to Rs 10,000/MWh].

In [ ]:
preds_df = pd.read_csv(TEST_PREDS_PATH)
preds_df['timestamp'] = pd.to_datetime(preds_df['timestamp'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# LightGBM
axes[0].scatter(preds_df['mcp_rs_per_mwh'], preds_df['pred_lightgbm'], alpha=0.25, color='#27AE60', s=12)
axes[0].plot([0, 11000], [0, 11000], color='#D9534F', ls='--', lw=2, label='Ideal Parity (y=y_hat)')
axes[0].axvline(10000, color='gray', ls=':', label='CERC Cap (Rs 10k)')
axes[0].axhline(10000, color='gray', ls=':')
axes[0].set_title('LightGBM: Actual vs. Predicted MCP (Test 2025)', fontweight='bold')
axes[0].set_xlabel('Actual MCP (Rs/MWh)')
axes[0].set_ylabel('Predicted MCP (Rs/MWh)')
axes[0].legend()

# XGBoost
axes[1].scatter(preds_df['mcp_rs_per_mwh'], preds_df['pred_xgboost'], alpha=0.25, color='#8E44AD', s=12)
axes[1].plot([0, 11000], [0, 11000], color='#D9534F', ls='--', lw=2, label='Ideal Parity (y=y_hat)')
axes[1].axvline(10000, color='gray', ls=':', label='CERC Cap (Rs 10k)')
axes[1].axhline(10000, color='gray', ls=':')
axes[1].set_title('XGBoost: Actual vs. Predicted MCP (Test 2025)', fontweight='bold')
axes[1].set_xlabel('Actual MCP (Rs/MWh)')
axes[1].set_ylabel('Predicted MCP (Rs/MWh)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4. Hourly Error Profile Across 24 Hours
Evaluating error distribution across diurnal periods, with special focus on the critical evening peak (18:00 - 22:00).

In [ ]:
preds_df['hour'] = preds_df['timestamp'].dt.hour
preds_df['err_lgb'] = np.abs(preds_df['mcp_rs_per_mwh'] - preds_df['pred_lightgbm'])
preds_df['err_xgb'] = np.abs(preds_df['mcp_rs_per_mwh'] - preds_df['pred_xgboost'])

hourly_mae = preds_df.groupby('hour')[['err_lgb', 'err_xgb']].mean()

plt.figure(figsize=(11, 5))
plt.plot(hourly_mae.index, hourly_mae['err_lgb'], marker='^', color='#27AE60', lw=2.2, label='LightGBM GBDT')
plt.plot(hourly_mae.index, hourly_mae['err_xgb'], marker='d', color='#8E44AD', lw=2.0, label='XGBoost GBDT')
plt.axvspan(18, 22, color='red', alpha=0.1, label='Evening Peak (18-22h)')
plt.title('Hourly Mean Absolute Error (MAE) Profile Across 24 Hours', fontweight='bold')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('MAE (Rs/MWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.tight_layout()
plt.show()